In [ ]:
#IMPORTS

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
from pyspark.sql.functions import col ,lower, trim ,to_timestamp ,desc, count, to_date ,broadcast
 


In [ ]:
#TASK1

spark = SparkSession.builder \
    .appName("CSV Data Processing Pipeline") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created")

In [ ]:
#TASK2

#Schema Definiton
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("transaction_timestamp", TimestampType(), True),
    StructField("platform", StringType(), True),
    StructField("country", StringType(), True)
])


# Reading the file
df_raw = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .schema(transaction_schema) \
    .csv("transactions.csv")


# Malformed rows
invalid_df = df_raw.filter(
    col("transaction_id").isNull() |
    col("user_id").isNull() |
    col("product_id").isNull() |
    col("category").isNull() |
    col("price").isNull() |
    col("quantity").isNull() |
    col("transaction_timestamp").isNull() |
    col("platform").isNull() |
    col("country").isNull()
)

# Valid rows
valid_df = df_raw.filter(
    col("transaction_id").isNotNull() &
    col("user_id").isNotNull() &
    col("product_id").isNotNull() &
    col("category").isNotNull() &
    col("price").isNotNull() &
    col("quantity").isNotNull() &
    col("transaction_timestamp").isNotNull() &
    col("platform").isNotNull() &
    col("country").isNotNull()
)

# Reporting
total_count = df_raw.count()
valid_count = valid_df.count()
invalid_count = invalid_df.count()

print("Total Records:", total_count)
print("Valid Records:", valid_count)
print("Invalid / Malformed Records:", invalid_count)

Total Records: 214
Valid Records: 197
Invalid / Malformed Records: 17


In [ ]:
#TASK3


# Drop missing rows
required_cols = ["transaction_id", "user_id", "product_id", "category", "price", "quantity", "transaction_timestamp"]
clean_df = valid_df.dropna(subset=required_cols)


#Drop Duplicates
clean_df = clean_df.dropDuplicates()


#Column Formatting and Type Casting
clean_df = clean_df.withColumn("category", trim(lower(col("category"))))
clean_df = clean_df.withColumn("price", col("price").cast("double")) \
                   .withColumn("quantity", col("quantity").cast("int")) \
                   .withColumn("transaction_timestamp",
                               to_timestamp(col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss"))


In [ ]:
#TASK4

clean_df.rdd.getNumPartitions()
df_repart = clean_df.repartition(4)
clean_df.cache()
final_df = clean_df.coalesce(1)

In [ ]:
#TASK5

total_revenue=clean_df.groupby("country").agg({"price":"sum"})
top_perfomer=clean_df.groupby("product_id").agg({"price":"sum"}).orderBy(desc("sum(price)")).show(5)
avg_ord_pltform=clean_df.groupby("platform").agg({"price":"avg"}).orderBy(desc("avg(price)")).show()


# Add transaction_date column by extracting date from transaction_timestamp
daily_trend_df = clean_df.withColumn("transaction_date", to_date(clean_df.transaction_timestamp))


daily_trend_df = daily_trend_df.groupBy("transaction_date") \
    .agg(count("transaction_id").alias("daily_transaction_count")) \
    .orderBy("transaction_date").show()

+----------+----------+
|product_id|sum(price)|
+----------+----------+
|   PROD076|   3999.98|
|   PROD046|   3599.98|
|   PROD092|   3399.98|
|   PROD086|   3199.98|
|   PROD032|   3199.98|
+----------+----------+
only showing top 5 rows
+--------+------------------+
|platform|        avg(price)|
+--------+------------------+
|     Web|  602.352551020408|
|  Mobile|309.26635416666676|
+--------+------------------+

+----------------+-----------------------+
|transaction_date|daily_transaction_count|
+----------------+-----------------------+
|      2024-01-15|                      7|
|      2024-01-16|                      9|
|      2024-01-17|                     10|
|      2024-01-18|                     10|
|      2024-01-19|                      9|
|      2024-01-20|                     10|
|      2024-01-21|                     10|
|      2024-01-22|                      8|
|      2024-01-23|                     10|
|      2024-01-24|                     10|
|      2024-01-25|  

In [ ]:
# TASK6

product_df=spark.read.csv("products.csv",header=True)
joined_df = clean_df.join(
    broadcast(product_df),
    on="product_id",
    how="left"
).show()

+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+-------------+
|product_id|transaction_id| user_id|     category|  price|quantity|transaction_timestamp|platform|country|        product_name|        brand|
+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+-------------+
|   PROD032|        TXN032|USER1032|  electronics|1599.99|       1|  2024-01-18 09:15:30|     Web| France|Gaming Console Ne...|     GameZone|
|   PROD082|        TXN182|USER1182|      fashion|  175.0|       1|  2024-02-02 09:15:30|     Web| France|         Blouse Silk|   ElegantTop|
|   PROD067|        TXN067|USER1067|home & garden| 245.99|       2|  2024-01-21 14:22:35|  Mobile| France|    Towel Set Cotton|   BathLuxury|
|   PROD001|        TXN101|USER1101|  electronics| 599.99|       1|  2024-01-25 08:30:45|  Mobile|    USA|Wireless Noise Ca...|    TechSound|
|   PR

In [ ]:
# TASK7

invalid_counter = spark.sparkContext.accumulator(0)

def count_invalid(row):
    if (
        row.transaction_id is None or
        row.user_id is None or
        row.product_id is None
    ):
        invalid_counter.add(1)

df_raw.foreach(count_invalid)
print("Total Invalid Records:", invalid_counter.value)

Total Invalid Records: 7


In [ ]:
# TASK8

final_df = clean_df.withColumn("transaction_date", to_date(col("transaction_timestamp")))
final_df.write \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet("output/partitioned_transactions")